<a href="https://colab.research.google.com/github/ChaiP-1205/Buying-Window-Intelligence-System/blob/main/Pinnacle_CSB_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata

genai.configure(api_key=userdata.get("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.6-flash")
response = model.generate_content("Say 'API key works' and nothing else.")
print(response.text)

API key works


In [ ]:
!pip install -q google-generativeai

In [ ]:
"""
CSB Incident Intelligence for Pinnacle Reliability - Gemini + Colab version
============================================================================
Extracts buying-signal intelligence from US Chemical Safety Board incident
reports and scores each by relevance to Pinnacle's QRO product.

Uses Gemini 3.6 Flash native PDF support - no text extraction step needed.
Runs in Google Colab with zero local setup.

QUICKSTART (5 minutes):
    1. Open https://colab.research.google.com and create a new notebook.
    2. Get a free Gemini API key at https://aistudio.google.com/app/apikey
    3. In Colab: click the key icon on the left sidebar, add secret named
       GEMINI_API_KEY, paste your key, and toggle "Notebook access" ON.
    4. Paste this entire file into a Colab cell.
    5. Manually upload 5 CSB PDF files into the Colab file browser
       (folder icon on left) - name them incident_1.pdf through incident_5.pdf.
       Get PDFs from https://www.csb.gov/investigations/ - pick recent ones.
    6. Run the cell.
"""

import os
import re
import csv
import json
from pathlib import Path
from datetime import datetime, date

# ---------------------------------------------------------------------------
# EDIT THIS: list the PDFs you uploaded and what you know about each
# ---------------------------------------------------------------------------
INCIDENTS = [
    {"file": "incident_1.pdf", "hint": "BP-Husky Oregon Refinery, OH (2022)"},
    {"file": "incident_2.pdf", "hint": "Husky Superior Refinery, WI (2018)"},
    {"file": "incident_3.pdf", "hint": "ExxonMobil Baton Rouge Refinery, LA (2016)"},
    {"file": "incident_4.pdf", "hint": "PES Philadelphia Refinery, PA (2019)"},
    {"file": "incident_5.pdf", "hint": "Dow Louisiana Operations, LA (2023)"},
    {"file": "incident_6.pdf", "hint": "PEMEX Deer Park, TX (2024)"},
    {"file": "incident_7.pdf", "hint": "KMCO LLC Crosby, TX (2019)"},
    {"file": "incident_8.pdf", "hint": "Wacker Polysilicon Charleston, TN (2020)"},
]

# ---------------------------------------------------------------------------
# The extraction prompt - tuned to Pinnacle's QRO product
# ---------------------------------------------------------------------------
EXTRACTION_PROMPT = """You are analyzing a US Chemical Safety Board (CSB) incident investigation report to generate GTM signals for Pinnacle Reliability. Pinnacle sells Quantitative Reliability Optimization (QRO) - software (Newton) that continuously models Probability of Failure at the failure-mode level, updating dynamically as new inspection, process, and maintenance data arrives.

Read the attached PDF and return ONLY valid JSON matching this exact schema. No markdown fences, no preamble, no explanation outside the JSON.

{
  "operator_parent_company": "string - the parent operating company",
  "facility_name": "string - specific facility or plant name",
  "facility_city_state": "string - City, State",
  "incident_date": "YYYY-MM-DD or null",
  "incident_type": "Explosion | Fire | Toxic Release | Runaway Reaction | Loss of Containment | Other",
  "equipment_involved": ["list of specific equipment - e.g. Fluid Catalytic Cracking Unit, Heat Exchanger, Storage Tank"],
  "primary_failure_mode": "string - specific technical failure mode (e.g. Sulfidation Corrosion, Stress Corrosion Cracking, Overpressure, Instrumentation Failure)",
  "root_cause_summary": "string - 2 sentences from the report's stated findings",
  "casualties_fatalities": integer,
  "casualties_injuries": integer,
  "regulatory_actions_mentioned": ["list of OSHA/EPA/other actions mentioned"],
  "would_qro_have_helped": boolean,
  "qro_relevance_reasoning": "string - 2-3 sentences explaining whether continuous, data-driven PoF modeling at the failure-mode level would have flagged this specific failure ahead of time. Be specific about the failure mode."
}

Rules:
- Use only information present in the PDF.
- Use JSON null (not the string "null") if a field is not stated.
- casualties default to 0 if not mentioned.
- Do NOT invent financial figures or dates.
- Return exactly one JSON object. Nothing else.
"""


# ---------------------------------------------------------------------------
# Gemini call - uses native PDF upload, no text extraction needed
# ---------------------------------------------------------------------------
def call_gemini_with_pdf(pdf_path: str, prompt: str) -> str:
    """Send a PDF to Gemini as inline data. Bypasses File API which fails
    with newer AQ.* format API keys."""
    import google.generativeai as genai

    api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get("GEMINI_API_KEY")
        except Exception:
            pass
    if not api_key:
        raise RuntimeError(
            "GEMINI_API_KEY not found. Add it as a Colab secret with "
            "Notebook access ON."
        )

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-3.6-flash")

    # Read PDF as bytes and send inline in the request
    pdf_bytes = Path(pdf_path).read_bytes()
    size_mb = len(pdf_bytes) / (1024 * 1024)
    if size_mb > 20:
        raise ValueError(f"PDF is {size_mb:.1f}MB, exceeds 20MB inline limit")

    response = model.generate_content([
        prompt,
        {"mime_type": "application/pdf", "data": pdf_bytes}
    ])
    return response.text


# ---------------------------------------------------------------------------
# Robust JSON parser - handles all the ways an LLM might malform output
# ---------------------------------------------------------------------------
def parse_json_response(raw: str) -> dict:
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip())
    raw = re.sub(r"\s*```$", "", raw.strip())
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found:\n{raw[:500]}")
    return json.loads(match.group(0))


# ---------------------------------------------------------------------------
# Buying-window urgency scoring - arbitrary weights, calibrate in prod
# ---------------------------------------------------------------------------
def compute_urgency_score(data: dict) -> int:
    score = 0

    # Recency: 0-30 pts
    if data.get("incident_date"):
        try:
            inc_date = datetime.strptime(data["incident_date"], "%Y-%m-%d").date()
            years_ago = (date.today() - inc_date).days / 365.25
            score += max(0, int(30 - years_ago * 4))
        except (ValueError, TypeError):
            pass

    # Severity: 0-30 pts
    fatalities = data.get("casualties_fatalities") or 0
    injuries = data.get("casualties_injuries") or 0
    score += min(30, fatalities * 10 + injuries * 1)

    # QRO product fit: 0-40 pts
    if data.get("would_qro_have_helped") is True:
        score += 40

    return score


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------
def run():
    results = []
    for i, incident in enumerate(INCIDENTS, 1):
        print(f"\n[{i}/{len(INCIDENTS)}] {incident['hint']}")
        pdf_path = incident["file"]

        if not Path(pdf_path).exists():
            print(f"  SKIP: {pdf_path} not found - upload it to Colab first")
            continue

        try:
            raw = call_gemini_with_pdf(pdf_path, EXTRACTION_PROMPT)
            data = parse_json_response(raw)
        except Exception as e:
            print(f"  FAILED: {e}")
            continue

        data["urgency_score"] = compute_urgency_score(data)
        data["source_file"] = pdf_path
        results.append(data)

        print(f"  score={data['urgency_score']}  "
              f"qro_fit={data.get('would_qro_have_helped')}  "
              f"mode={data.get('primary_failure_mode')}")

    if not results:
        print("\nNo results. Check PDFs are uploaded and API key is set.")
        return

    results.sort(key=lambda x: x["urgency_score"], reverse=True)

    # Flatten lists for CSV
    for r in results:
        for k in ("equipment_involved", "regulatory_actions_mentioned"):
            if isinstance(r.get(k), list):
                r[k] = "; ".join(str(x) for x in r[k])

    with open("pinnacle_csb_signals.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        w.writeheader()
        w.writerows(results)

    Path("pinnacle_csb_signals.json").write_text(json.dumps(results, indent=2))

    print("\n" + "=" * 72)
    print("TOP BUYING-WINDOW CANDIDATES FOR PINNACLE OUTBOUND")
    print("=" * 72)
    for r in results[:3]:
        print(f"\n[Score {r['urgency_score']}]  {r.get('operator_parent_company')}")
        print(f"  Facility:      {r.get('facility_name')} - {r.get('facility_city_state')}")
        print(f"  Failure mode:  {r.get('primary_failure_mode')}")
        print(f"  QRO fit:       {r.get('qro_relevance_reasoning')}")

    print(f"\nSaved: pinnacle_csb_signals.csv, pinnacle_csb_signals.json")


# In Colab you'll just call run() from a cell.
# If running as a script:
if __name__ == "__main__":
    run()




[1/8] BP-Husky Oregon Refinery, OH (2022)
  score=38  qro_fit=False  mode=Liquid Overfill

[2/8] Husky Superior Refinery, WI (2018)
  score=70  qro_fit=True  mode=Erosion of Spent Catalyst Slide Valve

[3/8] ExxonMobil Baton Rouge Refinery, LA (2016)
  score=4  qro_fit=False  mode=Inadvertent Disassembly of Pressure-Retaining Components

[4/8] PES Philadelphia Refinery, PA (2019)
  score=47  qro_fit=True  mode=Accelerated Hydrofluoric Acid Corrosion

[5/8] Dow Louisiana Operations, LA (2023)


ERROR:tornado.access:503 POST /v1beta/models/gemini-3.6-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 17298.96ms


  score=57  qro_fit=True  mode=Foreign Debris Impact Rupture Disc Puncture

[6/8] PEMEX Deer Park, TX (2024)
  score=52  qro_fit=False  mode=Inadvertent Opening of Active Equipment / Equipment Misidentification

[7/8] KMCO LLC Crosby, TX (2019)
  score=70  qro_fit=True  mode=Brittle Overload Fracture

[8/8] Wacker Polysilicon Charleston, TN (2020)
  score=19  qro_fit=False  mode=Brittle Overload

TOP BUYING-WINDOW CANDIDATES FOR PINNACLE OUTBOUND

[Score 70]  Husky Energy Inc.
  Facility:      Husky Superior Refinery - Superior, WI
  Failure mode:  Erosion of Spent Catalyst Slide Valve
  QRO fit:       The primary mechanical failure mode contributing to the disaster was severe erosion of the spent catalyst slide valve orifice and disc, which had experienced up to 85 percent metal loss in prior turnarounds and was normalized over 5-year operating cycles. Continuous, dynamic PoF modeling via QRO would have integrated historical inspection findings, operational differential pressure trend